In [62]:
import numpy as np
from typing import Type, Tuple
import mpmath as mp
import sympy as sp

In [33]:
def bisection(prec: int, a: float, b: float, err: float, func) -> Tuple[float, int]:
    with mp.workdps(prec):
        a = mp.mpf(a)
        b = mp.mpf(b)
        split = mp.mpf(0)
        
        n = int(mp.ceil(mp.log((b-a)/err)/mp.log(2)))
        j = 0
        for i in range(n):
            split = a + (b-a)/2
            if abs(func(split)) < err:
                break
            if mp.sign(func(a)) != mp.sign(func(split)):
                b = split
            else:
                a = split
            j+=1
        
        return split, j
        

In [34]:
f = (lambda x:mp.cos(x) * mp.cosh(x) - 1)
g = (lambda x:1/x-mp.tanh(x))
h = (lambda x:mp.power(2, -x) + mp.exp(x) + 2*mp.cos(x) - 6)

a = 3/2*mp.pi
b = 2*mp.pi

mp.nprint(bisection(10000, a, b, 1e-10, f), 10000)

(4.730040744862961284369442034019197487119783029907438276495668105781078338623046875, 31)


In [36]:
def Newton(prec: int, a: float, b: float, err: float, func_expr) -> float:
    with mp.workprec(prec):
        x = sp.symbols('x')
        deriv_expr = sp.diff(func_expr, x)
        
        f = sp.lambdify(x, func_expr, 'mpmath')
        df = sp.lambdify(x, deriv_expr, 'mpmath')
        
        x_n = mp.mpf(b)
        
        i = 0
        
        while True:
            fx = f(x_n)
            if abs(fx) < err:
                return x_n, i
            x_n = x_n - fx / df(x_n)
            i+=1
        

In [37]:
#f = (lambda x:mp.cos(x) * mp.cosh(x) - 1)
g = (lambda x:1/x-mp.tanh(x))
h = (lambda x:mp.power(2, -x) + mp.exp(x) + 2*mp.cos(x) - 6)

x = sp.symbols('x')

f = sp.cos(x)*sp.cosh(x)-1
g = 1/x-sp.tanh(x)
h = 2**(-x)+sp.exp(x) + 2*sp.cos(x) - 6

a = 3/2*mp.pi
b = 2*mp.pi

mp.nprint(Newton(10000, a, b, 1e-10, f), 10000)

(4.7300407448627755503324401725948864243615376209707116529020355187171638975044078302454032190011532900775989770032530257872188537341789388559772661062609381025631386107745222382701421617220885846533675730953260083087466122150828828200766777944634473281306342772380144416422397529215263121991176450217900991421335642800284408129636450684452047040952807285875586254438520423450590837176842126522958577647611934344254422288059023823162937230788177581223723373496240830560988126614887669403920769960138459130503820829172438278819678658611361138017142466434182771634374234292810830661611220297921880869455105740076987906309703104013689753480368710011345946406242440572093562514630164980362637987215730129477499202905873624967362892453338165326624893231980505846273311686939740353863696544766067461864461325023363169860458257428814524612433841831412923290790849473499609259572026972340622267025085998982325646143743928717426671596487489691886367541595617456763774833551874368552188970869280604364164075934

In [59]:
def Secant(prec: int, a: float, b: float, err: float, func) -> Tuple[float, int]:
    with mp.workprec(prec):
        a, b = mp.mpf(a), mp.mpf(b)
        
        i = 0
        
        while i < 1000:
            fa, fb = func(a), func(b)
            den = func(b)-func(a)
            
            if abs(den) < mp.mpf('1e-50'):
                raise ZeroDivisionError
            
            c = b - fb * (b-a)/den
            
            if abs(c-b) < err:
                return c, i
            
            a, b = b, c
            
            i+=1
        

In [60]:
f = (lambda x:mp.cos(x) * mp.cosh(x) - 1)
g = (lambda x:1/x-mp.tanh(x))
h = (lambda x:mp.power(2, -x) + mp.exp(x) + 2*mp.cos(x) - 6)

a = 3/2*mp.pi+1e-10
b = 2*mp.pi

mp.nprint(Secant(10000, a, b, 1e-10, f), 10000)

(4.7300407448627040714596097923886546819821750325236692732367144662518833229323644029833699408182183659621923451260804991455926202201189040353484860695571539156909839188523746113392525550815217530387342112844097012833093683478575748317578607731769335433686055848713569338906686854824738527266262540035060840748788466606661164545589267207922079231862028959356313341218376213749259522997029474023481266137831532979891998584076493841193987426138238974130099011193398718916979230370516670589759704886653021228436352339721693516596568026607066551418374550557924110183316771575557483445921239049775416881885396054723308168944063525293322113419612913092719271418750061856858518736715087108309198029170300811295760057006406141914704032300241507301796604662011657512134175229891936685356830690681029748348319911910166691973322000150690058443237974121338955717398354867554393310289515786872280138782732504382924322407027130945495743558248752437689237316447951672264357455027935262795200642696384866290635670041

In [67]:
def hybrid(prec: int, a: float, b: float, err: float, func) -> Tuple[float, int]:
    with mp.workdps(prec):
        a = mp.mpf(a)
        b = mp.mpf(b)
        
        j = 0
        for i in range(3):
            split = a + (b-a)/2
            if abs(func(split)) < err:
                break
            if mp.sign(func(a)) != mp.sign(func(split)):
                b = split
            else:
                a = split
            j+=1
        
        
        while j < 1000:
            fa, fb = func(a), func(b)
            den = func(b)-func(a)
            
            if abs(den) < mp.mpf('1e-50'):
                raise ZeroDivisionError
            
            c = b - fb * (b-a)/den
            
            if abs(c-b) < err:
                return c, j
            
            a, b = b, c
            
            j+=1
        

In [68]:
f = (lambda x:mp.cos(x) * mp.cosh(x) - 1)
g = (lambda x:1/x-mp.tanh(x))
h = (lambda x:mp.power(2, -x) + mp.exp(x) + 2*mp.cos(x) - 6)

a = 3/2*mp.pi+1e-10
b = 2*mp.pi

mp.nprint(hybrid(10000, a, b, 1e-10, f), 10000)

(4.7300407448627040260240484151250333728217429446282105734612836841013551188008682147640429272447333852575692393085324226641690090265844618932504310884296447138299128104644882136733974553780154624666885448877158781706675574213854924306626266937392712488593289390513253129963277263073089013096133317376049427940680391452655895803357005093522500969582028389502767326051805123657530662279395005555716800766006479171383829687706820320713827729832913551974009655895119051970992135323756481684692041286052744787902363684473829326614065503929453993910490018747001802338980307589377979583371562481768340527970838092137890495807158869903648597541708209034859858093847691593737834960675601669491992713115197241328343910744776652446864378871039730006925669042635715272995083267605733264438043889026837646196619889993154038424726042652119092214399862191843682592356091268421345381382721309051307473589445602378587997065574616957371648985982784191342295891485954563208585800708604510780531205744684704525138041631